# 02 · Validación del dataset maestro y construcción del rezago escolar

**Proyecto:** Brecha digital y rendimiento escolar en el Perú — ENAHO + ESCALE 2025  
**Fase:** 2 de la preparación de datos · notebook EJECUTABLE, no es una plantilla vacía.

**Primero:** termina de ejecutar `01_limpieza_enaho_escale_2025.ipynb` (la versión actualizada que incluye **P308C**).  
**Después:** abre este archivo dentro de `notebooks/` y ejecuta **Kernel → Restart & Run All**.

Este notebook utiliza **únicamente** el CSV maestro generado por la limpieza y conserva una sola tabla de estudiantes. No usa archivos adicionales ni descarga datos. Los casos sin datos educativos suficientes permanecerán **sin etiqueta**, nunca se rellenarán como `0`.

**Advertencia metodológica:** la regla de rezago es una *definición operativa* para el proyecto, no una medición oficial de tasas nacionales. Para informes poblacionales harán falta ponderaciones/diseño muestral. La guía solicita definir una variable objetivo y evitar fuga de información durante las fases de ML.

## Paso 1 — Localizar el proyecto y cargar el maestro

Comprobamos la ruta, las columnas indispensables, el año, la unicidad por persona y los seis dígitos de UBIGEO. Si falta `P308C`, termina el notebook 01 actualizado antes de continuar.

In [2]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display

cwd = Path.cwd().resolve()
candidatos = []
for base in (cwd, *cwd.parents):
    candidatos.extend((base, base / "proyecto_brecha_digital_2025"))
ROOT = next((p for p in candidatos if (p / "data" / "processed" / "dataset_enaho_escale_2025.csv").is_file()), None)
if ROOT is None:
    raise FileNotFoundError(
        f"No veo el maestro desde: {cwd}. Ejecuta notebook 01 y abre notebook 02 desde "
        "la carpeta proyecto_brecha_digital_2025/notebooks."
    )
RUTA = ROOT / "data" / "processed" / "dataset_enaho_escale_2025.csv"
REPS = ROOT / "reportes"
REPS.mkdir(parents=True, exist_ok=True)

# Todas las llaves se mantienen como cadenas, con ceros a la izquierda.
df = pd.read_csv(RUTA, dtype="string", encoding="utf-8-sig", keep_default_na=False)
for col in df.columns:
    df[col] = df[col].str.strip().replace("", pd.NA)

LLAVES = ["AÑO", "CONGLOME", "VIVIENDA", "HOGAR", "CODPERSO", "UBIGEO"]
NECESARIAS = LLAVES + ["EDAD", "P306", "P307", "P308A", "P308B", "P308C", "P314A", "P1121", "P1144"]
faltantes = sorted(set(NECESARIAS) - set(df.columns))
if faltantes:
    raise ValueError("Faltan columnas: " + ", ".join(faltantes) + ". Vuelve a ejecutar el notebook 01 actualizado.")
assert df["AÑO"].eq("2025").all(), "El dataset contiene años diferentes a 2025."
assert df["UBIGEO"].notna().all() and df["UBIGEO"].str.fullmatch(r"\d{6}").all(), "UBIGEO incorrecto."
assert not df[LLAVES].isna().any().any(), "Hay llaves de persona incompletas."
assert not df.duplicated(LLAVES).any(), "Hay personas duplicadas."

print("✓ Ruta del proyecto:", ROOT)
print("✓ Archivo maestro:", RUTA.name)
print("✓ Dimensiones iniciales:", df.shape)
print("✓ Duplicados de personas: 0")
print("✓ P308C encontrado: SÍ")


✓ Ruta del proyecto: C:\Users\Russbell\Downloads\inteligenciaartificial\datamodulos\data_limpia
✓ Archivo maestro: dataset_enaho_escale_2025.csv
✓ Dimensiones iniciales: (23645, 74)
✓ Duplicados de personas: 0
✓ P308C encontrado: SÍ


## Paso 2 — Revisar códigos de las variables educativas

El proyecto considera:

- `P306`: matrícula (1 = sí; 2 = no).
- `P307`: asistencia (1 = sí; 2 = no).
- `P308A`: nivel (2 = primaria; 3 = secundaria).
- `P308C`: grado de primaria (1 a 6).
- `P308B`: año de secundaria (1 a 5).
- `P314A`: uso de internet, conservado como código original.

Los códigos de vivienda e internet deben documentarse conforme a sus diccionarios antes de utilizarlos como variables explicativas en los modelos. **No confundimos `P308B` (año) con `P308C` (grado).**

In [3]:
VARIABLES_AUDITAR = ["P306", "P307", "P308A", "P308B", "P308C", "P314A", "P1144", "P1121"]
filas = []
for v in VARIABLES_AUDITAR:
    for codigo, frecuencia in df[v].fillna("<SIN DATO>").value_counts(dropna=False).items():
        filas.append({"VARIABLE": v, "CODIGO": str(codigo), "FRECUENCIA": int(frecuencia)})
auditoria_codigos = pd.DataFrame(filas)
display(auditoria_codigos)
print("Cruce nivel/curso (P308A y P308C):")
display(pd.crosstab(df["P308A"].fillna("NULO"), df["P308C"].fillna("NULO")))
auditoria_codigos.to_csv(REPS / "02_frecuencias_codigos_2025.csv", index=False, encoding="utf-8-sig")


,VARIABLE,CODIGO,FRECUENCIA
0,P306,1,20278
1,P306,2,3340
2,P306,<SIN DATO>,27
3,P307,1,17992
4,P307,<SIN DATO>,3367
5,P307,2,2286
6,P308A,2,10108
7,P308A,3,9382
8,P308A,<SIN DATO>,3367
9,P308A,1,391


Cruce nivel/curso (P308A y P308C):


P308C,1,2,3,4,5,6,NULO
P308A,,,,,,,
1,0,0,0,0,0,0,391
2,1477,1615,1712,1728,1741,1835,0
3,0,0,0,0,0,0,9382
4,0,0,0,0,0,0,144
5,0,0,0,0,0,0,233
7,1,1,1,3,1,1,12
NULO,0,0,0,0,0,0,3367


## Paso 3 — Definir quién puede recibir la etiqueta

**Población elegible:** persona de 6 a 17 años, matriculada **y** asistente, cursando primaria o secundaria regular con curso interpretable.

**Regla operativa:**

- Primaria: edad normativa = grado + 5. Ej.: 3.º de primaria → 8 años.
- Secundaria: edad normativa = año + 11. Ej.: 2.º de secundaria → 13 años.
- `REZAGO_ESCOLAR = 1` si edad actual es **mayor** que edad normativa; `0` en caso contrario.
- Fuera del universo elegible = `NA` (no se fuerza etiqueta).

Esta regla debe justificarse en la metodología, incluida la limitación de que la edad a la fecha de entrevista puede no coincidir con la edad al inicio del año escolar.

In [4]:
edad = pd.to_numeric(df["EDAD"], errors="coerce")
curso_prim = pd.to_numeric(df["P308C"], errors="coerce")
curso_sec = pd.to_numeric(df["P308B"], errors="coerce")

es_prim = df["P308A"].eq("2").fillna(False) & curso_prim.between(1, 6).fillna(False)
es_sec = df["P308A"].eq("3").fillna(False) & curso_sec.between(1, 5).fillna(False)
matriculado = df["P306"].eq("1").fillna(False)
asistente = df["P307"].eq("1").fillna(False)
elegible = (es_prim | es_sec) & matriculado & asistente & edad.between(6, 17).fillna(False)

nivel_ebr = pd.Series(pd.NA, index=df.index, dtype="string")
grados = pd.Series(pd.NA, index=df.index, dtype="Int64")
edad_normativa = pd.Series(pd.NA, index=df.index, dtype="Int64")
rezago = pd.Series(pd.NA, index=df.index, dtype="Int64")

nivel_ebr.loc[es_prim] = "Primaria"
nivel_ebr.loc[es_sec] = "Secundaria"
grados.loc[es_prim] = curso_prim.loc[es_prim].astype("Int64")
grados.loc[es_sec] = curso_sec.loc[es_sec].astype("Int64")
edad_normativa.loc[es_prim] = grados.loc[es_prim] + 5
edad_normativa.loc[es_sec] = grados.loc[es_sec] + 11
rezago.loc[elegible] = (edad.loc[elegible] > edad_normativa.loc[elegible]).astype("Int64")

df["NIVEL_EBR"] = nivel_ebr
df["GRADO_EBR"] = grados
df["EDAD_NORMATIVA"] = edad_normativa
df["REZAGO_ESCOLAR"] = rezago

assert (df["REZAGO_ESCOLAR"].notna().to_numpy(dtype=bool) == elegible.to_numpy(dtype=bool, na_value=False)).all(), "El objetivo se asignó fuera del universo elegible."
assert df["REZAGO_ESCOLAR"].dropna().isin([0, 1]).all(), "Etiquetas fuera de rango 0/1."
print("✓ Estudiantes en maestro:", len(df))
print("✓ Con etiqueta válida:", int(elegible.sum()))
print("✓ Sin etiqueta (se conservan):", int((~elegible).sum()))
print("✓ Rezago = 1:", int((rezago == 1).sum()))
print("✓ Rezago = 0:", int((rezago == 0).sum()))


✓ Estudiantes en maestro: 23645
✓ Con etiqueta válida: 17249
✓ Sin etiqueta (se conservan): 6396
✓ Rezago = 1: 7117
✓ Rezago = 0: 10132


## Paso 4 — Revisar la distribución del objetivo y faltantes

Los recuentos son descriptivos del archivo (sin aplicar factores de expansión); **no** son una tasa nacional oficial.

In [5]:
resumen = pd.crosstab(
    df.loc[elegible, "NIVEL_EBR"], df.loc[elegible, "REZAGO_ESCOLAR"],
    margins=True, margins_name="TOTAL"
)
print("Distribución de rezago por nivel educativo:")
display(resumen)
reporte_nulos = (df.isna().mean().mul(100).sort_values(ascending=False)
                 .rename("PORCENTAJE_NULOS").rename_axis("VARIABLE").reset_index())
print("Variables con mayor porcentaje de faltantes:")
display(reporte_nulos.head(15))
reporte_nulos.to_csv(REPS / "02_nulos_variables_2025.csv", index=False, encoding="utf-8-sig")


Distribución de rezago por nivel educativo:


REZAGO_ESCOLAR,0,1,TOTAL
NIVEL_EBR,,,
Primaria,5420,3535,8955
Secundaria,4712,3582,8294
TOTAL,10132,7117,17249


Variables con mayor porcentaje de faltantes:


,VARIABLE,PORCENTAJE_NULOS
0,P308C,57.217171
1,P316$7,27.354620
2,P316$4,27.354620
3,P316$3,27.354620
4,P316$2,27.354620
5,P316$8,27.354620
6,P314B$4,27.354620
7,P314B$5,27.354620
8,P314B$6,27.354620
9,P314B$7,27.354620


## Paso 5 — Variables TIC y prevención de fuga (anti-leakage)

Se conservan los códigos originales. Las conversiones técnicas `P1144` (0/1), `P1121` (0/1) y `P314A` (1/2) se agregan sin imputación. **La interpretación sustantiva de cada variable se debe contrastar con el diccionario del cuestionario INEI 2025 antes de publicarla.**

En el modelo supervisado de **rezago**, excluiremos como predictores la edad, el nivel, el grado y todas las variables usadas para construir la etiqueta. `CONGLOME` se usará para separar los grupos, no como predictor. La imputación y el escalamiento se ajustarán dentro de entrenamiento, no aquí.

In [6]:
def transformar_binaria(s, mapa):
    return s.map(mapa).astype("Int64")

df["INTERNET_HOGAR"] = transformar_binaria(df["P1144"], {"1": 1, "0": 0})
df["ELECTRICIDAD_HOGAR"] = transformar_binaria(df["P1121"], {"1": 1, "0": 0})
df["USO_INTERNET_MES"] = transformar_binaria(df["P314A"], {"1": 1, "2": 0})

EXCLUIR_DE_X = {
    "AÑO", "MES", "CONGLOME", "NCONGLOME", "VIVIENDA", "HOGAR", "CODPERSO", "UBIGEO",
    "EDAD", "P208A", "P306", "P307", "P308A", "P308B", "P308C",
    "NIVEL_EBR", "GRADO_EBR", "EDAD_NORMATIVA", "REZAGO_ESCOLAR", "FACTOR07"
}
print("Columnas finales:", len(df.columns))
print("Conglomerados distintos entre estudiantes etiquetados:", df.loc[elegible, "CONGLOME"].nunique())
print("Columnas excluidas de X para evitar fuga y memorizar identificadores:")
print(", ".join(sorted(EXCLUIR_DE_X)))
print("Nota: FACTOR07 se conserva para posibles ponderaciones, no como predictor.")


Columnas finales: 81
Conglomerados distintos entre estudiantes etiquetados: 4072
Columnas excluidas de X para evitar fuga y memorizar identificadores:
AÑO, CODPERSO, CONGLOME, EDAD, EDAD_NORMATIVA, FACTOR07, GRADO_EBR, HOGAR, MES, NCONGLOME, NIVEL_EBR, P208A, P306, P307, P308A, P308B, P308C, REZAGO_ESCOLAR, UBIGEO, VIVIENDA
Nota: FACTOR07 se conserva para posibles ponderaciones, no como predictor.


## Paso 6 — Guardar el maestro actualizado y el informe de control

El CSV maestro de la fase 1 se **actualiza** con las variables derivadas. No se crean dos datasets maestros divergentes. Si vuelves a ejecutar la limpieza 01, debes volver a ejecutar esta fase 02. Si tenías abierto el CSV en Excel, ciérralo antes de ejecutar esta celda.

In [7]:
assert not df.duplicated(LLAVES).any(), "Duplicados inesperados."
assert df["AÑO"].eq("2025").all()
assert len(df) == len(pd.read_csv(RUTA, usecols=["AÑO"], encoding="utf-8-sig"))

# Guarda una copia previa (si no existía) para recuperarte de errores futuros.
BACKUP = ROOT / "data" / "processed" / "dataset_enaho_escale_2025_antes_de_rezago.csv"
if not BACKUP.exists():
    import shutil
    shutil.copy2(RUTA, BACKUP)

df.to_csv(RUTA, index=False, encoding="utf-8-sig")
try:
    df.to_parquet(RUTA.with_suffix(".parquet"), index=False)
    print("✓ Parquet actualizado")
except (ImportError, TypeError, ValueError):
    print("Aviso: Parquet no actualizado (opcional; CSV guardado correctamente)")

reporte = {
    "fase": "02_validacion_objetivo_rezago", "anio": 2025,
    "registros_total": int(len(df)), "columnas": int(len(df.columns)),
    "duplicados_persona": int(df.duplicated(LLAVES).sum()),
    "universo_elegible": int(elegible.sum()), "fuera_del_universo": int((~elegible).sum()),
    "rezago_1": int((df["REZAGO_ESCOLAR"] == 1).sum()),
    "rezago_0": int((df["REZAGO_ESCOLAR"] == 0).sum()),
    "regla": "matriculado y asistente 6-17; primaria grado=P308C 1-6 edad normativa=grado+5, secundaria anio=P308B 1-5 edad normativa=anio+11; 1 si edad>normativa",
    "nota": "Regla operativa, no estimacion oficial. Revisar diccionario ENAHO 2025 y diseño muestral. Sin imputacion antes del split; futuros modelos deben dividir por CONGLOME.",
    "dataset_maestro": str(RUTA)
}
REPORTE = REPS / "02_control_objetivo_2025.json"
REPORTE.write_text(json.dumps(reporte, indent=2, ensure_ascii=False), encoding="utf-8")
print("✓ Dataset maestro actualizado:", RUTA)
print("✓ Copia anterior:", BACKUP)
print("✓ Informe de control:", REPORTE)
print("✓ Estado final:", df.shape, "; registros con etiqueta:", reporte["universo_elegible"])


✓ Parquet actualizado
✓ Dataset maestro actualizado: C:\Users\Russbell\Downloads\inteligenciaartificial\datamodulos\data_limpia\data\processed\dataset_enaho_escale_2025.csv
✓ Copia anterior: C:\Users\Russbell\Downloads\inteligenciaartificial\datamodulos\data_limpia\data\processed\dataset_enaho_escale_2025_antes_de_rezago.csv
✓ Informe de control: C:\Users\Russbell\Downloads\inteligenciaartificial\datamodulos\data_limpia\reportes\02_control_objetivo_2025.json
✓ Estado final: (23645, 81) ; registros con etiqueta: 17249


## Resultado y siguiente fase

Si la celda final indica **dataset actualizado** y **cero duplicados**, terminaste esta preparación. En los archivos disponibles, la ejecución de referencia produjo **23.645 registros, 81 columnas, 17.249 estudiantes con etiqueta de rezago (7.117 con rezago, 10.132 sin rezago)**. Si obtienes otros valores, no los sustituyas: revisa qué versión de la limpieza ejecutaste y comparte las salidas.

El siguiente paso será el **clustering** (al menos dos algoritmos; PCA y métricas de silueta/Davies–Bouldin) y luego el entrenamiento supervisado, con división por `CONGLOME`, seguido de SHAP; todo conforme a la guía del proyecto.